# R3-02 · IA generativa y LLMs — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional. Dos preguntas que todo servicio debería hacerse antes de publicar un texto escrito
por un LLM: **¿es fiel a la fuente?** y **¿se entiende?** Construirás medidas simples de
**fidelidad** (cobertura de palabras, números no respaldados) y de **legibilidad** (palabras
por oración), y las aplicarás a una reescritura de la introducción de la ficha de la PGU en **lectura fácil**.
Todo corre sin clave: las funciones que escribes se prueban sobre textos fijos.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, re, json, unicodedata, urllib.request
import pandas as pd

if not os.path.exists("fichas.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/fichas.csv", "fichas.csv")
df = pd.read_csv("fichas.csv")
pgu = df[df["id"] == 130457].iloc[0]
# Trabajamos con la introducción de la ficha (lo que va antes de la primera sección "## ").
fuente = pgu["texto"].split("\n## ")[0].strip()

STOP = {"de", "la", "el", "los", "las", "y", "en", "a", "con", "por", "para", "que", "se", "su",
        "sus", "un", "una", "del", "al", "o", "tu", "si", "mas", "es", "como", "desde", "hasta"}

def tokens_contenido(texto):
    """Palabras con contenido, en minúsculas y sin tildes (como en R3-01)."""
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return [w for w in re.findall(r"[a-z]+", t) if w not in STOP and len(w) >= 3]

# Dos resúmenes de la ficha de la PGU: uno fiel y uno con datos inventados.
RESUMEN_FIEL = ("Con la Reforma de Pensiones, la Pensión Garantizada Universal (PGU) subió. "
    "Desde febrero de 2026, si tienes 82 años o más recibes hasta $250.275, y si tienes entre 65 y 81 años, "
    "hasta $231.732. Desde septiembre de 2026, las personas de 75 años o más podrán recibir el monto máximo. "
    "Consulta con tu RUN cuándo aumentará tu PGU.")
RESUMEN_INFIEL = ("La PGU es un bono de $300.000 que se paga a todas las personas desde los 60 años. "
    "Para recibirla debes postular en tu municipalidad con tu Registro Social de Hogares.")
print(pgu["titulo"], "· introducción:", len(fuente.split()), "palabras")

La celda siguiente define `preguntar_llm()` (la misma de la lección). Sin clave, devuelve el
respaldo y todo funciona igual.

In [ ]:
# ── preguntar_llm(): el LLM (Gemini) detrás de UNA función ─────────────────────
# Cambiar de proveedor o de modelo = cambiar solo esta celda.
# En Colab: 🔑 (Secretos) → agrega GEMINI_API_KEY (gratis en aistudio.google.com).
import os

MODELO = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")

def _clave_gemini():
    try:                                   # Colab: secreto del notebook
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:                      # local: variable de entorno
        return os.environ.get("GEMINI_API_KEY")

def preguntar_llm(prompt, respaldo="[respuesta de respaldo — configura GEMINI_API_KEY para usar el LLM en vivo]"):
    """Envía el prompt a Gemini y devuelve el texto. Sin clave o sin red, devuelve
    `respaldo` para que el notebook siga corriendo (y se pueda verificar en CI)."""
    clave = _clave_gemini()
    if not clave:
        return respaldo
    try:
        from google import genai
        cliente = genai.Client(api_key=clave)
        return cliente.models.generate_content(model=MODELO, contents=prompt).text
    except Exception as e:
        print("⚠️ No se pudo llamar al LLM:", e)
        return respaldo

def get_llm():
    """Compatibilidad: devuelve una función generate(prompt)->str."""
    return preguntar_llm

## 1. Partir un texto en oraciones

Para medir legibilidad y fidelidad necesitamos **oraciones**. Escribe `oraciones(texto)`: parte después de `.`, `!` o `?` seguidos de espacio, y también en cada salto de línea; descarta los trozos vacíos. Pista: `re.split(r"(?<=[.!?])\s+|\n+", texto)`.

### ✍️ Ejercicio 1 — Escribe `oraciones`

In [ ]:
def oraciones(texto):
    # TODO: re.split con el patrón de la pista; quita espacios (strip) y descarta vacíos
    ...

print(oraciones(fuente))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert oraciones("Hola. ¿Cómo estás? Bien!\nFin") == ["Hola.", "¿Cómo estás?", "Bien!", "Fin"]
    assert oraciones("Monto: $250.275 al mes.") == ["Monto: $250.275 al mes."], "el punto de miles no corta"
    assert len(oraciones(fuente)) == 9
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "return [o.strip() for o in re.split(r'(?<=[.!?])\\s+|\\n+', texto) if o.strip()]")
    print("   Detalle:", e)

## 2. Cobertura: ¿de dónde salen las palabras del resumen?

Una medida simple de **fidelidad**: qué fracción de las palabras con contenido del resumen aparece también en la fuente. No prueba que el resumen sea correcto (puede reordenar palabras y cambiar el sentido), pero un valor bajo es una **alerta**.

### ✍️ Ejercicio 2 — Escribe `cobertura`

In [ ]:
def cobertura(resumen, fuente):
    del_resumen = tokens_contenido(resumen)
    de_la_fuente = set(tokens_contenido(fuente))
    if not del_resumen:
        return 0.0
    # TODO: fracción de palabras de del_resumen que están en de_la_fuente
    ...

print(cobertura(RESUMEN_FIEL, fuente), cobertura(RESUMEN_INFIEL, fuente))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert cobertura(fuente, fuente) == 1.0
    assert 0 <= cobertura(RESUMEN_INFIEL, fuente) < cobertura(RESUMEN_FIEL, fuente) <= 1
    assert cobertura("", fuente) == 0.0
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return sum(w in de_la_fuente for w in del_resumen) / len(del_resumen).')
    print("   Detalle:", e)

## 3. Números que no están en la fuente

La alerta más fuerte: un **número** (monto, edad, año) del resumen que no aparece en la fuente. Escribe `numeros_no_respaldados(resumen, fuente)`: extrae los números de ambos textos (con `numeros(texto)`, ya escrita, que quita los puntos de miles) y devuelve, ordenados, los del resumen que no están en la fuente.

### ✍️ Ejercicio 3 — Escribe `numeros_no_respaldados`

In [ ]:
def numeros(texto):
    return {n.replace(".", "") for n in re.findall(r"\d+(?:\.\d{3})*", texto)}

def numeros_no_respaldados(resumen, fuente):
    # TODO: números del resumen que NO están en la fuente, ordenados (resta de conjuntos + sorted)
    ...

print("fiel:  ", numeros_no_respaldados(RESUMEN_FIEL, fuente))
print("infiel:", numeros_no_respaldados(RESUMEN_INFIEL, fuente))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert numeros_no_respaldados(RESUMEN_FIEL, fuente) == []
    assert numeros_no_respaldados(RESUMEN_INFIEL, fuente) == ["300000", "60"]
    assert numeros_no_respaldados("Hasta $250.275 desde los 82", fuente) == []
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return sorted(numeros(resumen) - numeros(fuente)).')
    print("   Detalle:", e)

## 4. Lectura fácil: medir antes de reescribir

**Lectura fácil** es una forma de redactar para que todas las personas entiendan: frases cortas, una idea por oración, palabras comunes, siglas explicadas. Un indicador simple es el promedio de **palabras por oración**. Escríbelo y compáralo entre la ficha y una versión en lectura fácil (abajo verás el prompt que la pide al LLM).

### ✍️ Ejercicio 4 — Escribe `palabras_por_oracion`

In [ ]:
def palabras_por_oracion(texto):
    ors = oraciones(texto)
    # TODO: total de palabras (texto.split()) dividido por el número de oraciones; 0.0 si no hay oraciones
    ...

print("introducción original:", palabras_por_oracion(fuente))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert abs(palabras_por_oracion(fuente) - len(fuente.split()) / 9) < 1e-9
    assert palabras_por_oracion("Uno dos. Tres cuatro cinco seis.") == 3.0
    assert palabras_por_oracion("") == 0.0
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return len(texto.split()) / len(ors) if ors else 0.0.')
    print("   Detalle:", e)

## 5. Todo junto: pedir lectura fácil y auditarla

El guardrail revisa la fuente, el LLM reescribe (o se usa el respaldo) y tus tres medidas auditan el resultado. Si hay números no respaldados o la cobertura cae, el texto vuelve a revisión humana antes de publicarse.

In [ ]:
RESPALDO_LECTURA_FACIL = """La Pensión Garantizada Universal (PGU) subió.
El cambio es por la Reforma de Pensiones.
Desde febrero de 2026, el monto depende de tu edad.
Si tienes 82 años o más, recibes hasta $250.275.
Si tienes entre 65 y 81 años, recibes hasta $231.732.
Desde septiembre de 2026, desde los 75 años recibes hasta $250.275.
Consulta con tu RUN cuándo sube tu PGU."""

def prompt_lectura_facil(texto):
    return ("Reescribe este texto en lectura fácil para personas mayores.\n"
            "Reglas: una idea por oración; máximo 12 palabras por oración; palabras comunes; "
            "explica o evita las siglas; no agregues datos que no estén en el texto; "
            "conserva todos los montos, edades y fechas.\n\nTEXTO:\n" + texto)

PATRON_RUT = r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"      # guardrail de R3-02
if re.search(PATRON_RUT, fuente) is None:
    facil = preguntar_llm(prompt_lectura_facil(fuente), respaldo=RESPALDO_LECTURA_FACIL)
else:
    facil = ""
    print("⚠️ La fuente trae un RUT: no se envía al LLM.")
print(facil, "\n")
print(f"palabras por oración → original: {palabras_por_oracion(fuente):.1f} | lectura fácil: {palabras_por_oracion(facil):.1f}")
print("números no respaldados:", numeros_no_respaldados(facil, fuente),
      f"| cobertura: {cobertura(facil, fuente):.0%}")

## Para llevar

- Las medidas automáticas son **alertas**, no certificados: un resumen con 100 % de cobertura puede invertir el sentido de una frase.
- Los **números** son lo primero que se verifica: son lo que más daño hace si está mal.
- Lectura fácil se puede **medir** (palabras por oración) y **pedir** con reglas explícitas en el prompt — y luego se audita igual.

---

*Fin de la profundización de R3-02.*